# 00 — Setup, Stratified Splits, Data Validation, Reference Baselines & Evaluation Classifier

Implements **Section 12.1** (`00_setup_data_baselines.ipynb`) of the Generative Models Benchmark Specification:
1. Colab setup cell (mount Drive, install dependencies, print GPU & library versions, set seeds).
2. Create stratified splits (`SPLIT_SEED = 1234`) for MNIST and CIFAR-10, save to `splits/*.json`, and assert integrity.
3. Run the full data validation report (**Section 6.3, Checks V1–V14**) for both datasets.
4. Run Pre-flight Checks (**Section 14, Checks P1–P13**, including the 2-D Mixture of Gaussians test P6).
5. Compute and cache training-split Inception-v3 reference statistics on Drive keyed by `split_hash`.
6. Compute the **Ceiling** (real test images vs training reference) and **Floor** (full-covariance Gaussian on training pixels) reference rows (**Section 7.6, Checks P3 & P4**).
7. Train and evaluate the `SmallResNet` evaluation classifier once per dataset (**Section 7.7**).

In [ ]:
# 1. Colab setup cell (identical in every notebook per Section 3 & Section 12)
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/alisamadian/generative-models-benchmark.git'
REPO_DIR = Path('/content/generative-models-benchmark')
DRIVE_REPO_DIR = Path('/content/drive/MyDrive/generative-models-benchmark')

if 'google.colab' in sys.modules:
    from google.colab import drive
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')
    if DRIVE_REPO_DIR.exists() and (DRIVE_REPO_DIR / 'common').exists():
        os.chdir(DRIVE_REPO_DIR)
    else:
        if not REPO_DIR.exists():
            subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
        else:
            subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--rebase', '--autostash'], check=False)
        os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
else:
    # Local execution: ensure working directory is the repository root
    if Path.cwd().name == 'notebooks':
        os.chdir(Path.cwd().parent)
if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

# Reload common.* modules if already imported in this Python kernel after git pull
import importlib
for _mod_name in list(sys.modules.keys()):
    if _mod_name == 'common' or _mod_name.startswith('common.'):
        importlib.reload(sys.modules[_mod_name])

import torch
from common.config import SPLIT_SEED, protocol_dict, protocol_hash
from common.utils import library_versions, mount_drive, set_seed

DRIVE_PATHS = mount_drive()
set_seed(SPLIT_SEED)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f'GPU: {gpu_name} ({gpu_mem_gb:.2f} GB VRAM)')
else:
    print('GPU: None (running on CPU)')

print('Working directory:', Path.cwd())
print('Library versions:', library_versions())
print('Drive/Cache paths:', {k: str(v) for k, v in DRIVE_PATHS.items()})

## 2. Create Stratified Splits & Assert Integrity (Section 6.1, Check P1)

In [ ]:
from common.data import assert_split_integrity, create_splits, load_raw, load_splits

RAW_DATA = {}
SPLITS = {}

for ds in ('mnist', 'cifar10'):
    raw_tr_x, raw_tr_y, raw_te_x, raw_te_y = load_raw(ds, data_root='./data')
    RAW_DATA[ds] = (raw_tr_x, raw_tr_y, raw_te_x, raw_te_y)
    split_path = Path('splits') / f'{ds}_split.json'
    if not split_path.exists():
        split_data = create_splits(ds, raw_tr_y, save_path=split_path, raw_train_images=raw_tr_x)
    else:
        split_data = load_splits(ds, split_path=split_path)
        assert_split_integrity(split_data, dataset=ds, raw_train_labels=raw_tr_y)
    SPLITS[ds] = split_data
    print(f'[{ds.upper()}] split_hash={split_data["split_hash"]} | counts={split_data["counts"]}')

## 3. Full Data Validation Report (Section 6.3, Checks V1–V14 & Check P2)

In [ ]:
from common.data import validate_and_report_setup

DATA_CARDS = {}
for ds in ('mnist', 'cifar10'):
    DATA_CARDS[ds] = validate_and_report_setup(
        dataset=ds,
        raw_data=RAW_DATA[ds],
        split_data=SPLITS[ds],
        show_plots=True,
        figures_dir='./figures',
    )

## 4. Core Pre-Flight Checks (Section 14, Checks P6–P13)

In [ ]:
from common.preflight import (
    check_p6_toy_2d_mog,
    check_p7_nfe_accounting,
    check_p8_second_order_sanity,
    check_p9_ema,
    check_p10_checkpoint_resume,
    check_p12_uint8_pipeline,
    check_p13_test_access_guard,
)

print('P6 (Toy 2-D MoG):', check_p6_toy_2d_mog(steps=1200, save_fig_path='./figures/p6_toy_2d_mog.png', show_plot=True))
print('P7 (NFE Accounting):', check_p7_nfe_accounting()['passed'])
print('P8 (Second-Order Sanity):', check_p8_second_order_sanity())
print('P9 (EMA Warm-up & Invariance):', check_p9_ema())
print('P10 (Checkpoint & Resume):', check_p10_checkpoint_resume())
print('P12 (uint8 Pipeline):', check_p12_uint8_pipeline())
print('P13 (Test-Access Guard):', check_p13_test_access_guard())

## 5. Compute and Cache Training-Split Inception Reference Statistics (Section 10.1)

In [ ]:
from common.metrics import reference_stats

REF_STATS = {}
for ds in ('mnist', 'cifar10'):
    REF_STATS[ds] = reference_stats(
        dataset=ds,
        cache_dir=DRIVE_PATHS['fid_stats'],
        raw_data=RAW_DATA[ds],
        split_data=SPLITS[ds],
        include_val=True,
    )
    print(f'[{ds.upper()}] Cached reference stats for split_hash={REF_STATS[ds]["split_hash"][:16]}...')

## 6. Train and Evaluate the Evaluation Classifier Once (Section 7.7)

In [ ]:
from common.metrics import train_eval_classifier

CLASSIFIERS = {}
CLASSIFIER_INFO = {}
for ds in ('mnist', 'cifar10'):
    clf_ckpt = DRIVE_PATHS['checkpoints'] / ds / 'classifier' / 'best_classifier.pt'
    clf, info = train_eval_classifier(
        dataset=ds,
        save_path=clf_ckpt,
        raw_data=RAW_DATA[ds],
        split_data=SPLITS[ds],
    )
    CLASSIFIERS[ds] = clf
    CLASSIFIER_INFO[ds] = info
    print(f'[{ds.upper()} Classifier]: {info}')

## 7. Compute Reference Baselines (Ceiling & Floor) and Sanity Checks P3 & P4 (Section 7.6)

In [ ]:
from common.metrics import compute_ceiling_and_floor
from common.utils import write_results_json

for ds in ('mnist', 'cifar10'):
    base_res = compute_ceiling_and_floor(
        dataset=ds,
        ref_stats_dict=REF_STATS[ds],
        classifier=CLASSIFIERS[ds],
        raw_data=RAW_DATA[ds],
        split_data=SPLITS[ds],
    )
    # Cache test reference stats alongside training reference stats on Drive
    REF_STATS[ds]['test_mu'] = base_res['test_mu']
    REF_STATS[ds]['test_sigma'] = base_res['test_sigma']
    
    payload = {
        'run_id': f'{ds}_baselines',
        'dataset': ds,
        'model': 'baselines',
        'mode': 'setup',
        'seed': SPLIT_SEED,
        'protocol_hash': protocol_hash(),
        'split_hash': SPLITS[ds]['split_hash'],
        'classifier_info': CLASSIFIER_INFO[ds],
        'data_card': DATA_CARDS[ds],
        'metrics': {
            'ceiling': base_res['ceiling'],
            'floor': base_res['floor'],
            'white_noise_fid': base_res['white_noise_fid'],
            'sample_size_bias_p4': base_res['sample_size_bias_p4'],
        },
        'test_touched': True,
        'rerun_reason': 'setup_baseline_initialization',
    }
    out_file = Path('results/final') / f'{ds}_baselines.json'
    write_results_json(out_file, payload, overwrite_non_final=True)
    print(f'[{ds.upper()} Baselines]: Ceiling FID={base_res["ceiling"]["fid_train_ref"]:.2f}, '
          f'Floor FID={base_res["floor"]["fid_train_ref"]:.2f}, '
          f'White-Noise FID={base_res["white_noise_fid"]:.2f}')